In [5]:
import kagglehub

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode

from sklearn import model_selection, linear_model, tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler

init_notebook_mode(connected=True)

In [8]:
# Download latest version of Spotify Top 2000s Mega Dataset
path = kagglehub.dataset_download("iamsumat/spotify-top-2000s-mega-dataset")

# Loading datadataset & View
spotify_df = pd.read_csv(f"{path}/Spotify-2000.csv")

# Display the first 5 rows of the dataset
spotify_df.head(15)

,Index,Title,Artist,Top Genre,Year,Beats Per Minute (BPM),Energy,Danceability,Loudness (dB),Liveness,Valence,Length (Duration),Acousticness,Speechiness,Popularity
0,1,Sunrise,Norah Jones,adult standards,2004,157,30,53,-14,11,68,201,94,3,71
1,2,Black Night,Deep Purple,album rock,2000,135,79,50,-11,17,81,207,17,7,39
2,3,Clint Eastwood,Gorillaz,alternative hip hop,2001,168,69,66,-9,7,52,341,2,17,69
3,4,The Pretender,Foo Fighters,alternative metal,2007,173,96,43,-4,3,37,269,0,4,76
4,5,Waitin' On A Sunny Day,Bruce Springsteen,classic rock,2002,106,82,58,-5,10,87,256,1,3,59
5,6,The Road Ahead (Miles Of The Unknown),City To City,alternative pop rock,2004,99,46,54,-9,14,14,247,0,2,45
6,7,She Will Be Loved,Maroon 5,pop,2002,102,71,71,-6,13,54,257,6,3,74
7,8,Knights of Cydonia,Muse,modern rock,2006,137,96,37,-5,12,21,366,0,14,69
8,9,Mr. Brightside,The Killers,modern rock,2004,148,92,36,-4,10,23,223,0,8,77
9,10,Without Me,Eminem,detroit hip hop,2002,112,67,91,-3,24,66,290,0,7,82


In [9]:
spotify_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Index                   1994 non-null   int64
 1   Title                   1994 non-null   str  
 2   Artist                  1994 non-null   str  
 3   Top Genre               1994 non-null   str  
 4   Year                    1994 non-null   int64
 5   Beats Per Minute (BPM)  1994 non-null   int64
 6   Energy                  1994 non-null   int64
 7   Danceability            1994 non-null   int64
 8   Loudness (dB)           1994 non-null   int64
 9   Liveness                1994 non-null   int64
 10  Valence                 1994 non-null   int64
 11  Length (Duration)       1994 non-null   str  
 12  Acousticness            1994 non-null   int64
 13  Speechiness             1994 non-null   int64
 14  Popularity              1994 non-null   int64
dtypes: int64(11), str(4)
memory usag

In [10]:
spotify_df.describe()

,Index,Year,Beats Per Minute (BPM),Energy,Danceability,Loudness (dB),Liveness,Valence,Acousticness,Speechiness,Popularity
count,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.00000
mean,997.500000,1992.992979,120.215647,59.679539,53.238215,-9.008526,19.012036,49.408726,28.858074,4.994985,59.52658
std,575.762538,16.116048,28.028096,22.154322,15.351507,3.647876,16.727378,24.858212,29.011986,4.401566,14.35160
min,1.000000,1956.000000,37.000000,3.000000,10.000000,-27.000000,2.000000,3.000000,0.000000,2.000000,11.00000
25%,499.250000,1979.000000,99.000000,42.000000,43.000000,-11.000000,9.000000,29.000000,3.000000,3.000000,49.25000
50%,997.500000,1993.000000,119.000000,61.000000,53.000000,-8.000000,12.000000,47.000000,18.000000,4.000000,62.00000
75%,1495.750000,2007.000000,136.000000,78.000000,64.000000,-6.000000,23.000000,69.750000,50.000000,5.000000,71.00000
max,1994.000000,2019.000000,206.000000,100.000000,96.000000,-2.000000,99.000000,99.000000,99.000000,55.000000,100.00000


In [11]:
# Number of genres that have featured in the all time top 2000.

spotify_df['Top Genre'].value_counts()



Top Genre
album rock                 413
adult standards            123
dutch pop                   88
alternative rock            86
dance pop                   83
                          ... 
german alternative rock      1
classic italian pop          1
pop punk                     1
motown                       1
bebop                        1
Name: count, Length: 149, dtype: int64

In [12]:
#Converting Length (Duration) to an integer data type

print(f"Length (Duration) data type before: {spotify_df['Length (Duration)'].dtype}")

spotify_df.replace(',','', regex=True, inplace=True)
spotify_df['Length (Duration)']= spotify_df['Length (Duration)'].apply(pd.to_numeric,errors='coerce')

print(f"Length (Duration) data type after: {spotify_df['Length (Duration)'].dtype}")

Length (Duration) data type before: str
Length (Duration) data type after: int64


In [13]:
#Removing the Index column.
spotify_df.drop(columns = ['Index'], inplace = True)

#Converting all column titles to lowercase.
spotify_df.columns = map(str.lower, spotify_df.columns)

#Coverting column names to have no space between, if they do, replace space with an underscore "_"
spotify_df.rename(columns = {'top genre' : 'genre', 'beats per minute (bpm)':'beats_per_minute','loudness (db)': 'loudness','length (duration)': 'duration'}, inplace = True)

In [14]:
spotify_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   title             1994 non-null   str  
 1   artist            1994 non-null   str  
 2   genre             1994 non-null   str  
 3   year              1994 non-null   int64
 4   beats_per_minute  1994 non-null   int64
 5   energy            1994 non-null   int64
 6   danceability      1994 non-null   int64
 7   loudness          1994 non-null   int64
 8   liveness          1994 non-null   int64
 9   valence           1994 non-null   int64
 10  duration          1994 non-null   int64
 11  acousticness      1994 non-null   int64
 12  speechiness       1994 non-null   int64
 13  popularity        1994 non-null   int64
dtypes: int64(11), str(3)
memory usage: 218.2 KB


In [15]:
spotify_df.head(3)

,title,artist,genre,year,beats_per_minute,energy,danceability,loudness,liveness,valence,duration,acousticness,speechiness,popularity
0,Sunrise,Norah Jones,adult standards,2004,157,30,53,-14,11,68,201,94,3,71
1,Black Night,Deep Purple,album rock,2000,135,79,50,-11,17,81,207,17,7,39
2,Clint Eastwood,Gorillaz,alternative hip hop,2001,168,69,66,-9,7,52,341,2,17,69


Consolidating genre column